In [1]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
import xformers
import gc
gc.collect()
torch.cuda.empty_cache()

print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))
print("PyTorch Version:", torch.__version__)
print("xFormers Version:", xformers.__version__)

CUDA Available: True
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU
PyTorch Version: 2.6.0+cu124
xFormers Version: 0.0.29.post3


In [2]:
import os
import json
import re
import numpy as np
import torch
import gc
import evaluate
from tqdm import tqdm
from unsloth import FastLanguageModel

Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
C:\Users\Legion\AppData\Local\Temp\ipykernel_4228\1600695247.py:9: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
BASE_MODEL_DIR = "base_llama3_model"
ADAPTER_DIR = "adapters/AMI_Adapter"

print("Loading models for inference...")

# Load the Base Brain offline
if not os.path.exists(BASE_MODEL_DIR):
    raise FileNotFoundError(f"Local base model not found at '{BASE_MODEL_DIR}'. \n   Run the training script first to download and save the base model.")

Loading models for inference...


In [4]:
print(f"Loading Base Model OFFLINE from local folder: {BASE_MODEL_DIR}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL_DIR, 
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# ADAPTER LOADING
# base model remains frozen, only lora activated
if not os.path.exists(ADAPTER_DIR):
    raise FileNotFoundError(f"AMI Adapter not found at '{ADAPTER_DIR}'. \n   Make sure the adapter correctly saved.")

print(f"Loading AMI LoRA Adapter from: {ADAPTER_DIR}...")
model.load_adapter(ADAPTER_DIR)

# Optimize for fast inference
FastLanguageModel.for_inference(model)

Loading Base Model OFFLINE from local folder: base_llama3_model...
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading AMI LoRA Adapter from: adapters/AMI_Adapter...


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=3072, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=3072, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=307

In [5]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an intelligent AI meeting assistant. Analyze the transcript and extract the factual information.
CRITICAL RULES:
1. List EVERY single speaker who talks in the text (e.g., A, B, C). Do not miss any.
2. Extract Main decisions using bullet points (-).
3. Extract Action items using bullet points (-).
4. The Summary must be exactly 1 to 2 strictly factual sentence. Do NOT invent details. Do NOT write fake dialogue.

Format your output EXACTLY like this:
Thought: Key speakers: [List speakers]
Main decisions: 
- [point 1]
Action items: 
- [point 1]
Summary: [Final summary]

### Input:
{}

### Response:
{}"""


In [6]:
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]
print("Model fully loaded and ready for inference!")

Model fully loaded and ready for inference!


In [7]:
TEST_FILE = "phase1b_ami_test_gold.jsonl"
print(f"\nLoading test data from {TEST_FILE}...")

test_samples = []
with open(TEST_FILE, "r", encoding="utf-8") as f:
    for line in f:
        test_samples.append(json.loads(line))

results = []


Loading test data from phase1b_ami_test_gold.jsonl...


In [8]:
print("Agent is evaluating all meetings...")
for sample in tqdm(test_samples, desc="Evaluating All Meetings"): 
    unseen_meeting = sample["input"]
    gold_full_text = sample["output"] 
    
    # Truncation
    max_chars = 6000
    if len(unseen_meeting) > max_chars:
        truncated = unseen_meeting[:max_chars]
        last_period = truncated.rfind(".")
        unseen_meeting = truncated[:last_period + 1] if last_period != -1 else truncated
    
    inputs = tokenizer([alpaca_prompt.format(unseen_meeting, "Thought: Key speakers:")], return_tensors="pt").to("cuda")
    
    # THE ENGINE
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 150,       
        temperature = 0.1, # reduce hallucination         
        top_p = 0.9,                
        use_cache = True,
        repetition_penalty = 1.1, # prevents duplicates
        no_repeat_ngram_size = 0,   
        eos_token_id = terminators, 
    )
    
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    ai_full_text = generated_text.split("### Response:\n")[1].strip()
        
    results.append({"Generated_Full_Text": ai_full_text, "Gold_Full_Text": gold_full_text})

Agent is evaluating all meetings...


Evaluating All Meetings: 100%|██████████| 26/26 [04:40<00:00, 10.80s/it]


In [9]:
# regex based parser
def extract_sections(text):
    flags = re.IGNORECASE | re.DOTALL
    s_match = re.search(r'Key speakers:\s*(.*?)(?=\nMain decisions:|$)', text, flags=flags)
    d_match = re.search(r'Main decisions:\s*(.*?)(?=\nAction items:|$)', text, flags=flags)
    a_match = re.search(r'Action items:\s*(.*?)(?=\nSummary:|$)', text, flags=flags)
    sum_match = re.search(r'Summary:\s*(.*)', text, flags=flags)

    speakers = s_match.group(1).strip() if s_match and s_match.group(1).strip() else "[MISSING]"
    decisions = d_match.group(1).strip() if d_match and d_match.group(1).strip() else "[MISSING]"
    actions = a_match.group(1).strip() if a_match and a_match.group(1).strip() else "[MISSING]"
    summary = sum_match.group(1).strip() if sum_match and sum_match.group(1).strip() else "[MISSING]"

    return speakers, decisions, actions, summary

In [10]:
pred_speakers, pred_decisions, pred_actions, pred_summary = [], [], [], []
ref_speakers, ref_decisions, ref_actions, ref_summary = [], [], [], []

for res in results:
    p_spk, p_dec, p_act, p_sum = extract_sections(res["Generated_Full_Text"])
    r_spk, r_dec, r_act, r_sum = extract_sections(res["Gold_Full_Text"])
    
    pred_speakers.append(p_spk)
    pred_decisions.append(p_dec)
    pred_actions.append(p_act)
    pred_summary.append(p_sum)
    
    ref_speakers.append(r_spk)
    ref_decisions.append(r_dec)
    ref_actions.append(r_act)
    ref_summary.append(r_sum)

In [11]:
print("\nLoading BERTScore Model...")
bertscore = evaluate.load("bertscore")

def get_bert_f1(preds, refs):
    scores = bertscore.compute(predictions=preds, references=refs, lang="en")
    return np.mean(scores['f1']) * 100

f1_speakers = get_bert_f1(pred_speakers, ref_speakers)
f1_decisions = get_bert_f1(pred_decisions, ref_decisions)
f1_actions = get_bert_f1(pred_actions, ref_actions)
f1_summary = get_bert_f1(pred_summary, ref_summary)

print("\n" + "="*50)
print("GRANULAR SEMANTIC ACCURACY (BERTScore F1)")
print("="*50)
print(f"1. Key Speakers Accuracy:   {f1_speakers:.2f}%")
print(f"2. Main Decisions Accuracy: {f1_decisions:.2f}%")
print(f"3. Action Items Accuracy:   {f1_actions:.2f}%")
print(f"4. Summary Accuracy:        {f1_summary:.2f}%")
print("-" * 50)
overall_avg = (f1_speakers + f1_decisions + f1_actions + f1_summary) / 4
print(f"OVERALL PIPELINE SCORE:  {overall_avg:.2f}%")
print("="*50)


Loading BERTScore Model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



GRANULAR SEMANTIC ACCURACY (BERTScore F1)
1. Key Speakers Accuracy:   90.87%
2. Main Decisions Accuracy: 81.76%
3. Action Items Accuracy:   80.68%
4. Summary Accuracy:        81.62%
--------------------------------------------------
OVERALL PIPELINE SCORE:  83.73%


In [12]:
def evaluate_custom_meeting(meeting_text, title):
    print(f"\nEvaluating: {title}...\n" + "="*40)
    
    inputs = tokenizer([alpaca_prompt.format(meeting_text, "Thought: Key speakers:")], return_tensors = "pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 150,       
        temperature = 0.1,          
        top_p = 0.9,                
        use_cache = True,
        repetition_penalty = 1.1,   
        no_repeat_ngram_size = 0,   
        eos_token_id = terminators, 
    )

    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    response_only = generated_text.split("### Response:\n")[1].strip()

    # Safety net for clean cut-offs
    last_period = response_only.rfind(".")
    if last_period != -1:
        response_only = response_only[:last_period + 1]

    print(response_only)
    print("="*40)

In [13]:
# Meeting 1: Marketing Budget
meeting_1 = """
Speaker A: Alright everyone, let's look at the Q3 marketing budget. We are currently overspent by $5,000 on social media ads.
Speaker B: I can pause the new banner ad campaign for next week. That should save us about $2,000 right there.
Speaker C: If we shift our focus to organic SEO instead of paid clicks, we won't need the remaining $3,000 ad budget for November.
Speaker A: Great plan. Let's execute the ad pause immediately and pivot to SEO strategy by Friday.
"""

In [14]:
# Meeting 2: Smart Water Bottle
meeting_2 = """
Speaker A: Good morning everyone, and thank you for coming to the initial development meeting for our new smart water bottle project. Over the past two years, our company has grown steadily in the health and lifestyle market. Now we want to expand our product range with something innovative, practical, and appealing to modern consumers. Our objective is to design a smart water bottle that encourages healthy hydration habits while remaining affordable and profitable. Let’s begin.

Speaker B: I’ll provide insights into customer behavior, competitor products, pricing expectations, and current wellness trends.

Speaker C: I’ll focus on how users interact with the bottle and its connected app, ensuring it’s simple and intuitive.

Speaker D: I’ll handle the physical design, materials, internal components, and manufacturing feasibility.

Speaker A: Excellent. Let’s clarify our targets. The retail price should not exceed forty euros. Production cost must stay under eighteen euros to maintain our profit margin. The product must look modern, feel durable, and offer clear value beyond a regular water bottle. Innovation is important, but practicality and cost control are equally critical.

Speaker B: Consumers are increasingly health-conscious. Many people track steps, calories, and sleep, but hydration tracking is still underdeveloped. Existing smart bottles are often too expensive or overly complicated. Customers want something simple, reliable, and stylish.

Speaker A: So simplicity is key. Thoughts on user experience?

Speaker C: The bottle should automatically track water intake without requiring constant manual input. A sensor could detect how much water is consumed. The companion app should display daily goals, reminders, and progress in a clean, minimal interface. We should avoid overwhelming users with data.

Speaker D: From a design perspective, the bottle must be lightweight but sturdy. Stainless steel is durable but expensive. High-quality BPA-free plastic could reduce cost and weight. We should also consider insulation to keep drinks cold.

Speaker A: Insulation would add value, but we need to watch production costs. Should we offer two versions?

Speaker B: Launching with one strong core product might be safer. Too many versions at the start can confuse customers.

Speaker A: Good point. Let’s focus on one model first. What about appearance?

Speaker D: Many smart bottles look overly technical. Ours should resemble a premium everyday bottle. Clean lines, soft matte finish, and subtle LED indicators rather than bright screens.

Speaker C: Maybe a small light ring around the lid that glows gently to remind users to drink. No loud alarms—just subtle visual cues.

Speaker B: That would appeal to office workers and students. Discreet design is important.

Speaker A: What about customization?

Speaker D: We could offer interchangeable silicone sleeves in different colors. That allows personalization without changing the core structure.

Speaker B: Custom accessories could also create additional revenue streams.

Speaker A: Let’s talk functionality. How does it connect?

Speaker C: Bluetooth connection to a smartphone app. The app should send reminders based on activity levels and weather conditions. Pairing must be quick and stable.

Speaker D: Battery life is critical. Users won’t tolerate charging it every day. We should aim for at least one week of battery life.

Speaker A: Agreed. Reliability is essential. If tracking fails, customers lose trust immediately.

Speaker B: A dependable product will generate strong word-of-mouth marketing.

Speaker A: To summarize: we’re aiming for a smart water bottle priced at forty euros, with production under eighteen. It should feature automatic intake tracking, subtle LED reminders, a minimal app interface, durable but lightweight materials, long battery life, and optional colorful sleeves for personalization. The design must be modern, discreet, and user-friendly.

Let’s refine sketches and technical feasibility estimates next, then define our final concept direction and timeline.

"""

In [15]:
# Meeting 3: Electric Scooter
meeting_3 = """

Emma: Good afternoon everyone, and welcome to the first development meeting for our new electric scooter project. As urban transportation continues to evolve, we want to introduce a product that is practical, environmentally friendly, and affordable for everyday commuters. Our goal is to create a scooter that stands out in a competitive market while remaining cost-effective to produce. Let’s begin.

Emma: From Market Research, I’ll provide data on commuter behavior, competitor pricing, and current trends in urban mobility.

Noah: I’m responsible for User Experience and Interface Design. I’ll focus on how riders interact with the scooter, including the dashboard display and mobile app integration.

Ava: I’m the Industrial Designer. I’ll handle the physical structure, materials, battery placement, and overall form of the scooter.

Emma: Now, our targets: the scooter should retail for no more than three hundred euros. Production cost must stay under one hundred eighty euros to maintain a healthy margin. It must be reliable, safe, and visually appealing. We also want it to feel modern but not overly complicated.

Emma: Many current scooters are either too expensive or poorly built. Consumers want something durable with good battery range. Range anxiety is a major issue. At least twenty-five kilometers per charge would be competitive in this price range.

Noah: The dashboard should be minimal — speed, battery level, and distance traveled. No clutter. We can connect it to a smartphone app for detailed stats, ride history, and battery diagnostics. The app must be simple and quick to pair via Bluetooth.

Ava: From a design standpoint, weight is critical. If it’s too heavy, commuters won’t carry it onto buses or upstairs. Aluminum alloy could provide strength without excessive weight. It should also fold easily with a secure locking mechanism.

Emma: What about safety features?

Ava: Strong front and rear LED lights, reliable disc brakes, and a non-slip foot grip surface. Tires should handle both smooth pavement and slightly uneven city roads.

Emma: Consumers also care about charging time. If charging takes too long, it becomes inconvenient. Four to five hours maximum would be ideal.

Ava: Many scooters look identical. We could introduce subtle color accents along the frame — maybe interchangeable accent strips — while keeping the base matte black or silver.

Noah: We should avoid a complicated control panel. One multifunction button and a clear display would reduce confusion.

Emma: So, to summarize: retail price under three hundred euros, production under one hundred eighty. Minimum range of twenty-five kilometers, lightweight aluminum frame, foldable design, fast charging within five hours, simple dashboard with Bluetooth app support, strong safety features including LED lights and disc brakes, and subtle customizable color accents.

Emma: Let’s now develop initial design sketches and cost estimates, then confirm technical feasibility and outline a development timeline.

"""

In [16]:
# Meeting 4: Fitness Smartwatch
meeting_4 = """

Speaker A: Let’s focus on the new fitness smartwatch. Our main goal is to create something affordable but competitive. The retail price should stay under ninety euros.

Speaker B: If we want to stay under ninety, production needs to be below forty-five euros. That limits the type of display and materials we can use.

Speaker C: From a software perspective, we should prioritize core features: step tracking, heart rate monitoring, sleep tracking, and basic workout modes. We shouldn’t overload it.

Speaker D: Agreed. Users in this price range want simplicity. The interface should be clean, with swipe navigation and large icons.

Speaker E: Battery life is critical. Many smartwatches need daily charging. If we can offer at least seven days of battery life, that would be a strong selling point.

Speaker F: Design-wise, it should look slim and modern, not bulky. Interchangeable straps could allow personalization without increasing production costs too much.

Speaker A: What about water resistance?

Speaker E: It should be at least splash-proof, ideally suitable for swimming. That would increase appeal.

Speaker B: We need to check how that affects cost. Full waterproofing might push us over budget.

Speaker C: We can avoid complex third-party apps at launch. A simple companion app showing activity stats and progress charts will be enough.

Speaker D: Notifications from the phone are also important — calls and messages at minimum.

Speaker F: Two color options at launch — black and silver — to keep manufacturing simple.

Speaker A: So we’re agreeing on: price under ninety euros, production under forty-five, core fitness tracking features, seven-day battery life, water resistance if feasible, simple app integration, phone notifications, slim design, and interchangeable straps. Let’s move forward with feasibility estimates and initial design sketches.

"""

In [17]:
evaluate_custom_meeting(meeting_1, "Marketing Budget Meeting")


Evaluating: Marketing Budget Meeting...
Thought: Key speakers: A, B, C
Main decisions: 
- Shift focus from paid clicks to organic SEO
- Execute ad pause immediately
Action items: 
- Execute ad pause immediately
- Pivot to SEO strategy by Friday
Summary: The marketing budget will be shifted from paid clicks to organic SEO, with the ad pause executed immediately and the new strategy implemented by Friday.


In [18]:
evaluate_custom_meeting(meeting_2, "Smart Water Bottle Meeting")


Evaluating: Smart Water Bottle Meeting...
Thought: Key speakers: A, B, C, D
Main decisions: 
- Automatic water intake tracking
- Minimal app interface
- Subtle LED reminders
- Durable but lightweight materials
- Long battery life
- Optional colorful sleeves
Action items: 
- Refine sketches
- Technical feasibility estimates
- Final concept direction
Summary: Our smart water bottle will encourage healthy hydration habits through automatic tracking, minimal app interface, and subtle reminders, all within a modern, discreet, and user-friendly design. Its durable yet lightweight construction and long-lasting battery ensure reliability and affordability. The optional personalized sleeve adds a touch of style.


In [19]:
evaluate_custom_meeting(meeting_3, "Electric Scooter Meeting")


Evaluating: Electric Scooter Meeting...
Thought: Key speakers: Emma, Noah, Ava
Main decisions: 
- Battery type
- Charging method
- Safety features
Action items: 
- Finalize design specifications
- Develop prototype
- Conduct usability testing
Summary: Our new electric scooter will have a sleek, modern design with a focus on user experience, safety, and affordability. Its key features include a compact dashboard, Bluetooth connectivity, advanced safety features, and a unique folding mechanism. With a target retail price of three hundred euros and a production cost under one hundred eighty euros, our scooter aims to stand out in the competitive market while maintaining a healthy profit margin. The final design specifications, prototype development, and usability testing will help refine our product before its launch.


In [20]:
evaluate_custom_meeting(meeting_4, "Fitness Smartwatch Meeting")


Evaluating: Fitness Smartwatch Meeting...
Thought: Key speakers: A, B, C, D, E, F
Main decisions: 
- Seven-day battery life
- Water resistance
- Simple app integration
- Phone notifications
- Slim design
- Interchangeable straps
Action items: 
- Feasibility estimates
- Initial design sketches
Summary: The new fitness smartwatch should have a slim design, interchangeable straps, seven-day battery life, water resistance, simple app integration, phone notifications, and be priced under ninety euros. Its core features include step tracking, heart rate monitoring, sleep tracking, and basic workout modes. The user interface should be clean with swipe navigation and large icons. Production costs should be kept low by avoiding complex third-party apps and using simple companion apps.


In [22]:
# HARD SHUTDOWN & UNLOAD FROM GPU

import torch
import gc

print("Initiating VRAM Hard-Shutdown for Local Models...")

# 1. Track memory before cleanup
if torch.cuda.is_available():
    vram_before = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM Allocated Before: {vram_before:.2f} GB")

# 2. List of every heavy object that might be trapped in memory
heavy_objects = [
    'model', 'tokenizer', 'trainer', 'bertscore', 'rouge', 
    'dataset', 'train_dataset', 'eval_dataset', 'split_dataset',
    'inputs', 'outputs'
]

# 3. Delete them dynamically if they exist
for obj in heavy_objects:
    if obj in globals():
        print(f"Unloading '{obj}' from memory...")
        del globals()[obj]

# 4. Force aggressive Garbage Collection (Run twice to clear circular references)
gc.collect()
gc.collect()

# 5. Flush the GPU Cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # Clears memory shared between backend processes
    torch.cuda.synchronize()  # Waits for all GPU operations to completely finish
    
    # Track memory after cleanup
    vram_after = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM Allocated After:  {vram_after:.2f} GB")

print("\nGPU Memory Cleared. Your VRAM is now completely empty!")

Initiating VRAM Hard-Shutdown for Local Models...
VRAM Allocated Before: 2.30 GB
VRAM Allocated After:  2.30 GB

GPU Memory Cleared. Your VRAM is now completely empty!
